In [ ]:
import csv
import hashlib
import json
import os
import random
import re
import time
from datetime import datetime, timezone
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://www.newegg.com"

CATEGORY_URLS = {
    "laptop": "https://www.newegg.com/All-Laptop/SubCategory/ID-32",

}

OUT_DIR        = "./newegg_output"
OUT_CSV        = os.path.join(OUT_DIR, "laptops_newegg.csv")
CHECKPOINT     = os.path.join(OUT_DIR, "scraped_urls.txt")
URL_LIST_CACHE = os.path.join(OUT_DIR, "product_urls.txt")

MAX_PAGES_PER_CATEGORY = 20
LIMIT                  = None       
RESET_RUN              = True       

DELAY_SECONDS  = 4.0
DELAY_JITTER   = 3.0
REQUEST_TIMEOUT = 20
MAX_RETRIES     = 3

MAX_CONSECUTIVE_BLOCKS = 5
BLOCK_BACKOFF_SECONDS  = 90

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.4 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36 Edg/126.0.0.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
]
SESSION_USER_AGENT = random.choice(USER_AGENTS)

BLOCK_MARKERS = [
    "pardon our interruption", "access denied", "are you a human",
    "captcha", "request unsuccessful", "unusual traffic", "bot detection",
]

FIELD_MAP = {
    "cpu":     ["cpu type", "processor name", "processor", "cpu"],
    "ram":     ["memory", "ram", "system memory"],
    "storage": ["ssd", "hard drive", "storage", "hdd"],
    "gpu":     ["gpu/vpu", "graphics", "video card", "gpu"],
    "display": ["screen size", "display", "screen"],
    "battery": ["battery life", "battery"],
}

FIELDNAMES = [
    "title", "price", "price_is_suspicious", "url",
    "cpu", "ram", "storage", "gpu", "display", "battery",
    "rating", "review_count", "category", "description",
    "document_text", "reviews_json", "raw_specs_json",
    "field_provenance_json", "scraped_at", "html_sha256",
]


class BlockedError(Exception):
    pass


def build_session():
    session = requests.Session()
    retry = Retry(
        total=MAX_RETRIES,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update(session_headers())
    return session


def session_headers(referer=None):
    headers = {
        "User-Agent": SESSION_USER_AGENT,
        "Accept-Language": "en-US,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Upgrade-Insecure-Requests": "1",
        "Connection": "keep-alive",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def polite_sleep():
    time.sleep(DELAY_SECONDS + random.uniform(0, DELAY_JITTER))


def clean_text(s):
    if not s:
        return ""
    return re.sub(r"\s+", " ", s).strip()


def parse_price(value):
    if value is None:
        return ""
    text = str(value)
    text = re.sub(r"\s+", "", text)
    text = text.replace(",", "")
    match = re.search(r"(\d+(?:\.\d{1,2})?)", text)
    if not match:
        return ""
    try:
        return f"{float(match.group(1)):.2f}"
    except ValueError:
        return match.group(1)


def is_valid_price(price_str):
    try:
        return bool(price_str) and float(price_str) > 1.0
    except (ValueError, TypeError):
        return False


def looks_blocked(soup, resp_text):
    has_structure = bool(
        soup.select_one("h1.product-title")
        or soup.select("a.item-title")
        or soup.select_one("meta[property='og:title']")
    )
    if has_structure:
        return False

    visible_text = soup.get_text(" ").lower()
    if any(marker in visible_text for marker in BLOCK_MARKERS):
        return True
    if len(resp_text) < 20000:
        return True
    return False


def get_soup(url, session, referer=None):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            headers = {"Referer": referer} if referer else {}
            resp = session.get(url, headers=headers, timeout=REQUEST_TIMEOUT)

            if resp.status_code == 200:
                soup = BeautifulSoup(resp.text, "lxml")
                if looks_blocked(soup, resp.text):
                    raise BlockedError(f"Block page detected at {url}")
                return soup, resp.text

            if resp.status_code in (403, 429, 503):
                print(f"  [warn] possible block status {resp.status_code} "
                      f"(attempt {attempt}/{MAX_RETRIES})")
                if attempt == MAX_RETRIES:
                    raise BlockedError(f"HTTP {resp.status_code} at {url}")
            else:
                print(f"  [warn] status {resp.status_code} "
                      f"(attempt {attempt}/{MAX_RETRIES})")

        except BlockedError:
            raise
        except requests.RequestException as e:
            print(f"  [warn] request error: {e} (attempt {attempt}/{MAX_RETRIES})")

        time.sleep(3 * attempt)

    return None, None


def collect_product_urls(search_url, max_pages, session):
    urls, seen = [], set()

    for page in range(1, max_pages + 1):
        page_url = (f"{search_url}&page={page}" if "?" in search_url
                    else f"{search_url}?page={page}")
        print(f"[listing] page {page}: {page_url}")

        try:
            soup, _ = get_soup(page_url, session, referer=search_url)
        except BlockedError:
            print("  [blocked] listing page blocked — stopping early")
            break
        if soup is None:
            print("  [warn] could not load listing page — stopping")
            break

        cells = soup.select(
            "a.item-title, .item-info a.item-title, "
            ".product-info a.item-title, .item-container a.item-title"
        )
        if not cells:
            print("  [warn] no product links found — last page or selector stale")
            break

        new_this_page = 0
        for a in cells:
            href = a.get("href")
            if not href:
                continue
            full = urljoin(BASE_URL, href.split("?")[0])
            if full not in seen:
                seen.add(full)
                urls.append(full)
                new_this_page += 1

        print(f"  found {len(cells)} links, {new_this_page} new "
              f"(total {len(urls)})")
        if new_this_page == 0:
            break
        polite_sleep()

    return urls


def collect_all_urls(session):
    all_urls, seen = [], set()
    for category, url in CATEGORY_URLS.items():
        print(f"\n=== Collecting URLs for: {category} ===")
        urls = collect_product_urls(url, MAX_PAGES_PER_CATEGORY, session)
        added = 0
        for u in urls:
            if u not in seen:
                seen.add(u)
                all_urls.append(u)
                added += 1
        print(f"[{category}] {len(urls)} collected, {added} new unique")
    return all_urls


def backup_file(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d-%H%M%S")
        bak = f"{path}.bak-{ts}"
        os.replace(path, bak)
        print(f"[reset] backed up {path} -> {bak}")


def load_urls_from_file(path):
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    urls, seen = [], set()
    for chunk in re.split(r"(?=https?://)", text):
        chunk = chunk.strip()
        if not chunk.startswith("http"):
            continue
        url = chunk.split()[0].strip().rstrip(",").split("?")[0]
        if url and url not in seen:
            seen.add(url)
            urls.append(url)
    return urls


def load_checkpoint():
    return set(load_urls_from_file(CHECKPOINT))


def append_checkpoint(url):
    with open(CHECKPOINT, "a", encoding="utf-8") as f:
        f.write(url + "\n")


BAD_CLASS_RE = re.compile(
    r"(item-action|item-sponsored|sponsor|similar|recommend|carousel|"
    r"swiper|related|recently-viewed)", re.I
)


def _tag_has_bad_class(tag):
    if not hasattr(tag, "get"):
        return False
    classes = tag.get("class", [])
    if not classes:
        return False
    return bool(BAD_CLASS_RE.search(" ".join(classes)))


def inside_bad_container(tag):
    if _tag_has_bad_class(tag):
        return True
    for parent in tag.parents:
        if parent.name == "[document]":
            break
        if _tag_has_bad_class(parent):
            return True
    return False


def extract_price_from_strong(strong_tag):
    if not strong_tag:
        return ""
    dollars = re.sub(r"[^\d]", "", strong_tag.get_text())
    if not dollars:
        return ""
    sup = strong_tag.find_next_sibling("sup")
    if not sup:
        sup = strong_tag.find_next("sup")
    cents = re.sub(r"[^\d]", "", sup.get_text()) if sup else ""
    cents = (cents + "00")[:2]
    return f"{dollars}.{cents}"


def extract_price_from_element(el):
    if not el:
        return ""
    strong = el.find("strong")
    if strong:
        price = extract_price_from_strong(strong)
        if is_valid_price(price):
            return price
    return parse_price(el.get_text(" "))


def extract_main_dom_price(soup):

    for el in soup.select("[class*='price-current_']"):
        if inside_bad_container(el):
            continue
        price = extract_price_from_element(el)
        if is_valid_price(price):
            return price

    for container_sel in [".product-buy-box", ".product-action",
                          ".product-buying-options", "#ProductPrice"]:
        container = soup.select_one(container_sel)
        if not container:
            continue
        for el in container.select(".price-current, [class*='price-current_']"):
            if inside_bad_container(el):
                continue
            price = extract_price_from_element(el)
            if is_valid_price(price):
                return price

    return ""


def extract_jsonld_price(soup):
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue
        items = data if isinstance(data, list) else [data]
        for item in items:
            if not isinstance(item, dict):
                continue
            if item.get("@type") == "Product":
                offers = item.get("offers", {})
                if isinstance(offers, list):
                    offers = offers[0] if offers else {}
                if isinstance(offers, dict):
                    for key in ("price", "lowPrice"):
                        p = parse_price(offers.get(key))
                        if is_valid_price(p):
                            return p
    return ""


def extract_embedded_price(url, raw_html):

    if not raw_html:
        return ""

    parent_item = ""
    m = re.search(r"/p/([^/?#]+)", url)
    if m:
        parent_item = m.group(1)

    if parent_item:
        pattern = (r'"ParentItem"\s*:\s*"' + re.escape(parent_item) +
                   r'".{0,2000}?"FinalPrice"\s*:\s*([\d.]+)')
        m = re.search(pattern, raw_html, re.S)
        if m:
            price = parse_price(m.group(1))
            if is_valid_price(price):
                return price

    for pat in [r'"NewItemLowestPrice"\s*:\s*([\d.]+)',
                r'"FinalPrice"\s*:\s*([\d.]+)']:
        m = re.search(pat, raw_html)
        if m:
            price = parse_price(m.group(1))
            if is_valid_price(price):
                return price

    return ""


def extract_hidden_reviews(raw_html):

    reviews = []
    if not raw_html:
        return reviews

    comment_pattern = r'"Comments"\s*:\s*"((?:[^"\\]|\\.)*)"'
    for match in re.finditer(comment_pattern, raw_html):
        raw_comment = match.group(1)
        try:
            comment = raw_comment.encode("utf-8").decode("unicode_escape")
            comment = re.sub(r"<[^>]+>", "", comment).strip()
            if len(comment) > 30 and "javascript" not in comment.lower():
                reviews.append(comment)
        except Exception:
            continue

    return list(set(reviews))  # deduplicate


def extract_specs(soup):
    rows = []
    containers = soup.select(
        "#product-details, #Specs, .product-details, .tab-pane"
    )
    for c in containers:
        rows.extend(c.select("table tr"))
    if not rows:
        rows = soup.select("table.table-horizontal tr, div.tab-pane table tr")

    specs = {}
    for row in rows:
        cells = row.find_all(["th", "td"])
        if len(cells) >= 2:
            key = clean_text(cells[0].get_text())
            val = clean_text(cells[1].get_text())
            if key and val and len(key) < 100:
                specs[key] = val
    return specs


def is_bad_spec_match(field, candidate, raw_key):
    key = raw_key.lower()
    if field == "ram" and candidate == "memory":
        return any(w in key for w in
                   ["video", "vram", "max", "supported", "slot", "speed",
                    "type", "card"])
    if field == "display" and candidate in ("display", "screen"):
        return any(w in key for w in
                   ["type", "resolution", "touch", "aspect", "brightness",
                    "color", "panel", "refresh"])
    if field == "storage" and candidate == "ssd":
        return any(w in key for w in
                   ["interface", "type", "slot", "read", "write", "protocol"])
    return False


def extract_title_fallbacks(title):
    if not title:
        return {}
    t = title.lower()
    result = {}

    # RAM
    for pat in [r"(\d{1,3})\s*gb\s*(?:lpddr\d|ddr\d)",
                r"(\d{1,3})\s*gb\s+(?:ram|memory)"]:
        m = re.search(pat, t)
        if m:
            result["ram"] = f"{m.group(1)}GB"
            break

    # Storage
    m = re.search(r"(\d+(?:\.\d+)?)\s*tb\s*(?:ssd|nvme|pcie|storage)", t)
    if m:
        amt = float(m.group(1))
        result["storage"] = f"{int(amt)}TB" if amt == int(amt) else f"{amt}TB"
    else:
        m = re.search(r"(\d{2,4})\s*gb\s*(?:ssd|nvme|pcie|emmc|storage)", t)
        if m:
            result["storage"] = f"{m.group(1)}GB"

    # CPU
    for pat in [r"intel\s+core\s+ultra\s+\d\s+\d{3}[a-z]{0,3}",
                r"intel\s+core\s+i\d-\d{4}[a-z]{0,3}",
                r"intel\s+core\s+\d\s+\d{2,3}[a-z]{0,3}",
                r"intel\s+core\s+i\d",
                r"intel\s+processor\s+n\d{3}",
                r"amd\s+ryzen\s+ai\s+\d\s+\d{3}",
                r"amd\s+ryzen\s+\d\s+\d{4}[a-z]{0,3}",
                r"amd\s+ryzen\s+\d",
                r"apple\s+m\d(?:\s+pro|\s+max)?",
                r"qualcomm\s+snapdragon\s+x\s+(?:elite|plus)"]:
        m = re.search(pat, t)
        if m:
            result["cpu"] = m.group(0).title()
            break

    # GPU
    for pat in [r"geforce\s+rtx\s+\d{4}\s*(?:ti|super)?(?:\s+laptop\s+gpu)?",
                r"gtx\s+\d{4}\s*(?:ti|super)?",
                r"amd\s+radeon\s+rx\s+\d{4}[a-z]*",
                r"intel\s+arc\s+\d{3}[a-z]*",
                r"radeon\s+\d{3}[a-z]*",
                r"intel\s+iris\s+xe\s+graphics",
                r"amd\s+radeon\s+graphics",
                r"qualcomm\s+adreno(?:\s+gpu)?",
                r"adreno(?:\s+gpu)?"]:
        m = re.search(pat, t)
        if m:
            result["gpu"] = m.group(0).title()
            break

    # Display size
    m = re.search(r'(\d{1,2}(?:\.\d)?)\s*(?:"|in\b|inch)', t)
    if m:
        result["display"] = f'{m.group(1)}"'

    # Battery
    m = re.search(r"up to\s+(\d+)\s*hours?", t)
    if m:
        result["battery"] = f"Up to {m.group(1)} Hours"

    return result


def map_specs(specs, title=""):
    mapped, provenance = {}, {}
    lower_specs = {k.lower().strip(): v for k, v in specs.items()}
    title_fb = extract_title_fallbacks(title)

    for field, candidates in FIELD_MAP.items():
        value, matched_cand, matched_key = "", None, None

        for cand in candidates:
            if cand in lower_specs:
                value = lower_specs[cand]
                matched_cand, matched_key = cand, cand
                break

        if not value:
            for cand in candidates:
                for raw_key, raw_val in lower_specs.items():
                    if cand in raw_key and not is_bad_spec_match(field, cand, raw_key):
                        value = raw_val
                        matched_cand, matched_key = cand, raw_key
                        break
                if value:
                    break

        if not value and field in title_fb:
            value = title_fb[field]
            matched_cand, matched_key = "title_regex", "title"

        mapped[field] = value
        provenance[field] = (
            {"candidate": matched_cand, "raw_key": matched_key}
            if value else None
        )

    return mapped, provenance


def extract_description(soup):
    bullets = soup.select("ul.product-bullets li, .product-bullets li")
    if bullets:
        return " | ".join(clean_text(b.get_text()) for b in bullets
                          if clean_text(b.get_text()))
    for sel in ["#product-details", ".product-overview",
                ".product-description", ".description"]:
        tag = soup.select_one(sel)
        if tag:
            text = clean_text(tag.get_text(" "))
            if len(text) > 20:
                return text
    meta = soup.select_one("meta[name='description'], meta[property='og:description']")
    if meta and meta.get("content"):
        return clean_text(meta["content"])
    return ""


def extract_category(soup):
    crumbs = soup.select("div.breadcrumb a, .breadcrumbs a, div.breadcrumbs a")
    if crumbs:
        return " > ".join(clean_text(b.get_text()) for b in crumbs
                          if clean_text(b.get_text()))
    return ""


def scrape_product_page(url, session):
    soup, raw_html = get_soup(url, session, referer=BASE_URL)
    if soup is None:
        return None

    title_tag = soup.select_one("h1.product-title")
    title = clean_text(title_tag.get_text()) if title_tag else ""
    if not title:
        og = soup.select_one("meta[property='og:title']")
        if og and og.get("content"):
            title = clean_text(og["content"])

    price = extract_main_dom_price(soup)
    if not price:
        price = extract_jsonld_price(soup)
    if not price:
        price = extract_embedded_price(url, raw_html)

    try:
        price_float = float(price) if price else None
    except ValueError:
        price_float = None
    price_suspicious = "1" if (price_float is None or price_float < 100) else "0"

    specs = extract_specs(soup)
    mapped, provenance = map_specs(specs, title)

    description = extract_description(soup)

    reviews = extract_hidden_reviews(raw_html)

    rating = ""
    rating_tag = soup.select_one(".item-rating")
    if rating_tag:
        rating = rating_tag.get("title", "") or clean_text(rating_tag.get_text())
    review_count = ""
    rc_tag = soup.select_one(".item-rating-num")
    if rc_tag:
        review_count = clean_text(rc_tag.get_text())

    category = extract_category(soup)

    parts = []
    if title:
        parts.append(f"Product: {title}")
    if category:
        parts.append(f"Category: {category}")
    if price:
        parts.append(f"Price: ${price}")
    if specs:
        spec_str = "; ".join(f"{k}: {v}" for k, v in specs.items())
        parts.append(f"Specifications: {spec_str}")
    if description:
        parts.append(f"Description: {description}")
    if reviews:
        parts.append("Customer Reviews: " + " | ".join(reviews[:5]))
    document_text = "\n".join(parts)

    # --- Provenance ---
    scraped_at = datetime.now(timezone.utc).isoformat()
    html_sha256 = hashlib.sha256(raw_html.encode("utf-8")).hexdigest() if raw_html else ""

    return {
        "title": title,
        "price": price,
        "price_is_suspicious": price_suspicious,
        "url": url,
        "cpu": mapped["cpu"],
        "ram": mapped["ram"],
        "storage": mapped["storage"],
        "gpu": mapped["gpu"],
        "display": mapped["display"],
        "battery": mapped["battery"],
        "rating": rating,
        "review_count": review_count,
        "category": category,
        "description": description,
        "document_text": document_text,
        "reviews_json": json.dumps(reviews, ensure_ascii=False),
        "raw_specs_json": json.dumps(specs, ensure_ascii=False),
        "field_provenance_json": json.dumps(provenance, ensure_ascii=False),
        "scraped_at": scraped_at,
        "html_sha256": html_sha256,
    }


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    if RESET_RUN:
        print("[reset] Backing up old output files...")
        backup_file(OUT_CSV)
        backup_file(CHECKPOINT)

    session = build_session()
    print(f"Session User-Agent (fixed): {SESSION_USER_AGENT}")

    print("\n=== Step 1: collecting / loading product URLs ===")
    if os.path.exists(URL_LIST_CACHE):
        product_urls = load_urls_from_file(URL_LIST_CACHE)
        print(f"Loaded {len(product_urls)} URLs from cached {URL_LIST_CACHE}")
    else:
        product_urls = collect_all_urls(session)
        with open(URL_LIST_CACHE, "w", encoding="utf-8") as f:
            f.write("\n".join(product_urls))
        print(f"Total product URLs collected: {len(product_urls)}")

    if not product_urls:
        print("No product URLs found. Exiting.")
        return

    already_done = load_checkpoint()
    remaining = [u for u in product_urls
                 if urljoin(BASE_URL, u) not in already_done]
    print(f"{len(already_done)} already scraped, {len(remaining)} remaining")

    if LIMIT:
        remaining = remaining[:LIMIT]
        print(f"Limiting this run to {len(remaining)} products")

    print("\n=== Step 2: scraping product pages ===")
    consecutive_blocks = 0
    scraped_this_run = 0

    with open(OUT_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if f.tell() == 0:
            writer.writeheader()

        for i, url in enumerate(remaining, 1):
            full_url = urljoin(BASE_URL, url)
            print(f"[{i}/{len(remaining)}] {full_url}")

            try:
                row = scrape_product_page(full_url, session)
            except BlockedError as e:
                consecutive_blocks += 1
                print(f"  [BLOCKED] {e} "
                      f"({consecutive_blocks}/{MAX_CONSECUTIVE_BLOCKS})")
                if consecutive_blocks >= MAX_CONSECUTIVE_BLOCKS:
                    print(f"\n[stop] {MAX_CONSECUTIVE_BLOCKS} consecutive blocks. "
                          f"{scraped_this_run} saved. Re-run later to resume.")
                    break
                time.sleep(BLOCK_BACKOFF_SECONDS)
                continue

            if row is None:
                print("  [skip] page load failed (non-block)")
                polite_sleep()
                continue

            consecutive_blocks = 0
            writer.writerow(row)
            f.flush()
            append_checkpoint(full_url)
            scraped_this_run += 1
            polite_sleep()

    print(f"\nDone. {scraped_this_run} products scraped this run.")
    print(f"Output: {OUT_CSV}")
    print(f"Checkpoint: {CHECKPOINT}")


if __name__ == "__main__":
    main()

[reset] Backing up old output files...
Session User-Agent (fixed): Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36

=== Step 1: collecting / loading product URLs ===

=== Collecting URLs for: laptop ===
[listing] page 1: https://www.newegg.com/All-Laptop/SubCategory/ID-32?page=1
  found 36 links, 36 new (total 36)
[listing] page 2: https://www.newegg.com/All-Laptop/SubCategory/ID-32?page=2
  found 36 links, 36 new (total 72)
[listing] page 3: https://www.newegg.com/All-Laptop/SubCategory/ID-32?page=3
  found 36 links, 36 new (total 108)
[listing] page 4: https://www.newegg.com/All-Laptop/SubCategory/ID-32?page=4
  found 36 links, 36 new (total 144)
[listing] page 5: https://www.newegg.com/All-Laptop/SubCategory/ID-32?page=5
  found 36 links, 36 new (total 180)
[listing] page 6: https://www.newegg.com/All-Laptop/SubCategory/ID-32?page=6
  found 36 links, 36 new (total 216)
[listing] page 7: https://www.newegg.com/All-Laptop/SubCatego